# Tập 2.9 · Tổng kết Mùa 2: robot đi theo bạn 🚶🤖

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap29/tap29_di_theo.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 2 · ⏱ ~35 phút · Giáo trình: Principles of Robot Autonomy, Chương 6 + 10 (+ Chương 3)

Camera + YOLO (tập 2.8) + PI (tập 1.3) thành robot đi theo người. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video (30–60 giây).
2. **Tự viết** khâu đo khoảng cách, bộ PI và logic chọn trạng thái (3 bài), có ô chấm phát lại 320 bước của video.
3. Vẽ đường đi và khoảng cách.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap29")
sys.path[:0] = ["bai_tap", "sim", ".."]

import math
from kiem_tra import goc
g = goc()                          # chạy lại ../sim/di_theo.py (YOLO của tập 2.8 + 320 bước)
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
import numpy as np
thay = [r for r in g.LOG if r["z_hat"] is not None]
loi = [abs(r["z_hat"] - r["kc_that"]) for r in thay]
print(f"thấy người ở {len(thay)}/{len(g.LOG)} khung · sai số khoảng cách trung bình {np.mean(loi) * 100:.1f} cm")
print("trạng thái:", {s: sum(r["trang_thai"] == s for r in g.LOG) for s in ("theo", "dừng", "tìm")})

## 2. Bài tập

Khung YOLO `(lop, diem, cx, cy, w, h)` theo tỉ lệ ảnh 128 px; lop 0 = người, 2 = bóng.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `uoc_luong(d)` | người $Z = H F/(hN)$, bóng $Z = D F/(wN)$, hướng $=\arctan((c_x - 0{,}5)N/F)$ |
| **2** | `lenh(z_hat, huong, tich_phan)` | $e = Z - 2$; PI cho $v \in [0, 1{,}2]$; $\omega = -K_\theta\cdot$hướng; kẹp tích phân ±1,5 |
| **3** | `quyet_dinh(dets)` | bóng gần ngay trước → "dừng"; có người → "theo"; còn lại → "tìm" |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import math

N, F = 128, 60.0            # ảnh 128 px, tiêu cự 60 px
H_NGUOI, D_BONG = 1.7, 0.25  # người cao 1,7 m, bóng đường kính 0,25 m
R_KC = 2.0                   # khoảng cách muốn giữ (m)
DT = 0.1
KP, KI, KTH = 0.8, 0.15, 1.6
V_MAX, W_MAX = 1.2, 1.5


def uoc_luong(d):
    """Bài 1: từ một khung, ước lượng khoảng cách Z (m) và hướng (rad, dương = vật lệch sang phải ảnh).

    - Hướng = atan((cx − 0.5)·N / F)          (cx = 0.5 là giữa ảnh)
    - Người (lop 0): biết chiều cao thật nên Z = H_NGUOI·F / (h·N)     (Eq. 6.1 đảo ngược, h·N là chiều cao khung theo pixel)
    - Bóng (lop khác 0): dùng bề rộng: Z = D_BONG·F / (w·N)
    Trả về (Z, huong)."""
    raise NotImplementedError("TODO Bài 1: (Z, hướng)")


def lenh(z_hat, huong, tich_phan):
    """Bài 2: bộ điều khiển đi theo người (PI cho tốc độ tới, P cho tốc độ quay).

    e = z_hat − R_KC (dương là đang xa hơn mong muốn)
    tich_phan_moi = kẹp(tich_phan + e·DT, −1.5, 1.5)
    v = kẹp(KP·e + KI·tich_phan_moi, 0, V_MAX)        (không bao giờ lùi)
    w = kẹp(−KTH·huong, −W_MAX, W_MAX)                 (người lệch phải → quay phải, ω âm)
    Trả về (v, w, tich_phan_moi)."""
    raise NotImplementedError("TODO Bài 2: (v, w, tích phân mới)")


def quyet_dinh(dets):
    """Bài 3: chọn trạng thái từ danh sách khung YOLO của một khung hình.

    1. Có bóng (lop 2): lấy quả điểm cao nhất, uoc_luong; nếu Z < 1.6 m VÀ |hướng| < 30° thì "dừng".
    2. Không phải "dừng" mà có người (lop 0) thì "theo".
    3. Còn lại "tìm".
    Trả về một trong 3 chuỗi "dừng", "theo", "tìm"."""
    raise NotImplementedError("TODO Bài 3: trạng thái")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
ve(None, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Robot tưởng người cao 1,7 m. Nếu người thật cao 1,5 m hay 1,9 m thì robot thực ra giữ cách bao xa khi định giữ 2 m?

In [ ]:
for H_that in [1.5, 1.7, 1.9]:
    # khung cao h = F·H_that/Z_that; robot tính Z = F·1.7/h = Z_that·1.7/H_that, và giữ Z = 2 m
    print(f"người cao {H_that} m: robot giữ cách thật {2.0 * H_that / 1.7:.2f} m")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap29).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)